In [1]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [2]:
from transformers import AutoImageProcessor, ResNetForImageClassification
from transformers import ResNetConfig, ResNetModel

from datasets import load_dataset

In [10]:
import numpy as np
import pandas as pd

In [7]:
metapd = pd.read_csv("../data/ddidiversedermatologyimages/ddi_metadata.csv")

In [8]:
metapd = metapd.rename(columns={"DDI_file":'file_name'})

In [45]:
metapd['label'] = np.where(metapd["malignant"], 1, 0)

In [46]:
metapd.to_csv("../data/ddidiversedermatologyimages/metadata.csv", index=False)

In [4]:

# Initializing a ResNet resnet-50 style configuration
configuration = ResNetConfig()

# Initializing a model (with random weights) from the resnet-50 style configuration
model = ResNetModel(configuration)

# Accessing the model configuration
configuration = model.config

In [4]:
image_processor = AutoImageProcessor.from_pretrained("microsoft/resnet-50")


In [2]:

model = ResNetForImageClassification.from_pretrained("microsoft/resnet-50")


In [47]:
dataset = load_dataset("imagefolder", data_dir="../data/ddidiversedermatologyimages/")

Resolving data files:   0%|          | 0/659 [00:00<?, ?it/s]

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [48]:
dataset['train'][6]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=918x675>,
 'Unnamed: 0': 6,
 'DDI_ID': 7,
 'skin_tone': 56,
 'malignant': True,
 'disease': 'melanoma-acral-lentiginous',
 'label': 1}

In [39]:
from datasets import load_metric

metric = load_metric("accuracy")

In [49]:
dataset["train"].features["label"]

Value(dtype='int64', id=None)

In [41]:
# dataset = dataset.class_encode_column("label")

Casting to class labels:   0%|          | 0/656 [00:00<?, ? examples/s]

In [42]:
dataset['train'][6]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=918x675>,
 'Unnamed: 0': 6,
 'DDI_ID': 7,
 'skin_tone': 56,
 'malignant': True,
 'disease': 'melanoma-acral-lentiginous',
 'label': 1}

In [50]:
labels = (0, 1)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label



In [51]:
label2id

{0: 0, 1: 1}

In [52]:
from torchvision.transforms import (
    CenterCrop,
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomResizedCrop,
    Resize,
    ToTensor,
)

In [53]:
from torchvision import transforms

In [54]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'Unnamed: 0', 'DDI_ID', 'skin_tone', 'malignant', 'disease', 'label'],
        num_rows: 656
    })
})

In [55]:
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

composed_transforms = transforms.Compose([
                    transforms.Resize(299),
                    transforms.CenterCrop(299),
                    transforms.ToTensor(),
                    normalize])

In [56]:

def preprocess_train(example_batch):
    """Apply train_transforms across a batch."""
    example_batch["pixel_values"] = [
        composed_transforms(image.convert("RGB")) for image in example_batch["image"]
    ]
    return example_batch

In [57]:
dataset['train'][6]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=918x675>,
 'Unnamed: 0': 6,
 'DDI_ID': 7,
 'skin_tone': 56,
 'malignant': True,
 'disease': 'melanoma-acral-lentiginous',
 'label': 1}

In [58]:
splits = dataset["train"].train_test_split(test_size=0.1)

In [59]:
train_ds = splits['train']
val_ds = splits['test']

In [60]:
train_ds.set_transform(preprocess_train)

In [61]:
val_ds.set_transform(preprocess_train)

In [62]:
train_ds[0]

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=369x463>,
 'Unnamed: 0': 168,
 'DDI_ID': 169,
 'skin_tone': 56,
 'malignant': True,
 'disease': 'squamous-cell-carcinoma',
 'label': 1,
 'pixel_values': tensor([[[ 2.0263,  2.1119,  2.1462,  ...,  1.7865,  1.7352,  1.5125],
          [ 2.0948,  2.1119,  2.1290,  ...,  1.7009,  1.5982,  1.3413],
          [ 2.1119,  2.1290,  2.1633,  ...,  1.5297,  1.3755,  1.1529],
          ...,
          [-1.0048, -0.9192, -0.8335,  ..., -0.4397, -0.4911, -0.2513],
          [-0.9877, -0.9020, -0.7822,  ..., -0.4054, -0.4397, -0.2342],
          [-0.9877, -0.8335, -0.6623,  ..., -0.3883, -0.4568, -0.2856]],
 
         [[ 2.1134,  2.2710,  2.3410,  ...,  1.2031,  1.1331,  0.8529],
          [ 2.2535,  2.3060,  2.3235,  ...,  1.0805,  0.9405,  0.5903],
          [ 2.3235,  2.3410,  2.3235,  ...,  0.8704,  0.6954,  0.3803],
          ...,
          [-1.2304, -1.1604, -1.0553,  ..., -0.7577, -0.8627, -0.7577],
          [-1.2304, -1.1429, -0

In [63]:

model = ResNetForImageClassification.from_pretrained("microsoft/resnet-50", label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes = True,)


Some weights of ResNetForImageClassification were not initialized from the model checkpoint at microsoft/resnet-50 and are newly initialized because the shapes did not match:
- classifier.1.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.1.weight: found shape torch.Size([1000, 2048]) in the checkpoint and torch.Size([2, 2048]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [64]:
from transformers import TrainingArguments, Trainer

In [65]:
batch_size = 16
num_epochs = 3

args = TrainingArguments(
    "ttttttttName",
    remove_unused_columns=False,
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    warmup_ratio=0.1,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

/home/NETID/xiruod/anaconda3/envs/seamlessm4tv2/lib/python3.9/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [66]:
import numpy as np

# the compute_metrics function takes a Named Tuple as input:
# predictions, which are the logits of the model as Numpy arrays,
# and label_ids, which are the ground-truth labels as Numpy arrays.
def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [69]:
import torch

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    # labels = torch.tensor([([0,1] if  example["label"] else [1,0] ) for example in examples ])
    labels = torch.tensor([example["label"] for example in examples ])
    return {"pixel_values": pixel_values, "labels": labels}

In [70]:
trainer = Trainer(
    model,
    args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn,
)

In [71]:
train_results = trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
0,No log,0.690238,0.590909
1,0.702800,0.683829,0.651515
2,0.674400,0.681695,0.666667


In [94]:
for batch in trainer.get_train_dataloader():
    break

In [95]:
batch['pixel_values'].shape

torch.Size([16, 3, 299, 299])

In [96]:
batch['labels'].shape

torch.Size([16])

In [97]:
model(**batch)

ImageClassifierOutputWithNoAttention(loss=tensor(0.6613, device='cuda:0', grad_fn=<NllLossBackward0>), logits=tensor([[ 0.0563, -0.0382],
        [ 0.0568,  0.0337],
        [ 0.0923, -0.0653],
        [ 0.1360, -0.0276],
        [ 0.1668, -0.0309],
        [ 0.0254, -0.0214],
        [ 0.0970, -0.0133],
        [ 0.0924, -0.0662],
        [ 0.0705, -0.0286],
        [ 0.1579, -0.0625],
        [ 0.0748, -0.0414],
        [ 0.1556, -0.0646],
        [ 0.0892, -0.0492],
        [ 0.1167, -0.0060],
        [ 0.1282, -0.0468],
        [ 0.0573, -0.0532]], device='cuda:0', grad_fn=<AddmmBackward0>), hidden_states=None)